# Lab Work - 5.2

## Q1. Regularization

### 01. L2-Regularised (Ridge) Logistic Regression Objective

$$ J_{L2}(w) = -\frac{1}{n}\sum_{i=1}^n \Big[ y_i \log(\sigma(w^Tx_i)) + (1-y_i)\log(1-\sigma(w^Tx_i)) \Big] + \frac{\lambda}{2}\|w\|^2 $$

### 02. Gradient of L2 Objective

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def l2_logistic_gradient(X, y, w, lambda_reg=0.1):
    """Gradient of L2-regularized logistic loss"""
    n = X.shape[0]
    z = X @ w
    preds = sigmoid(z)
    grad = (1/n) * (X.T @ (preds - y)) + lambda_reg * w
    return grad

# Test
X = np.array([[1, -3], [1, 1], [1, 0], [1, 5]])  # bias term included
y = np.array([0, 1, 0, 1])
w = np.array([-1, 3, 0.5])
print("Regularized gradient:", l2_logistic_gradient(X, y, w, lambda_reg=0.1))

### 04. L1 (Lasso) Objective & Subgradients

$$ J_{L1}(w) = \text{cross-entropy} + \lambda \|w\|_1 $$

Non-differentiable at w=0 → use **subgradients** (sign function with 0 in [-1,1] at zero).

### 05. L1 vs L2 Comparison + Constraint Regions

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6,6))
theta = np.linspace(0, 2*np.pi, 200)
ax.plot(np.cos(theta), np.sin(theta), label='L2 (circle)', lw=2)
# L1 diamond
ax.plot([-1,0,1,0,-1], [0,1,0,-1,0], 'r--', label='L1 (diamond)', lw=2)
ax.set_aspect('equal')
ax.grid(True)
ax.legend()
ax.set_title('L1 vs L2 Constraint Regions in 2D')
plt.show()

### 05. sklearn C parameter

`C = 1/λ`. `C=0.01` → **strong regularization**, `C=100` → **weak regularization**.

## Q2. Multiclass Classification

In [ ]:
def softmax(z):
    """Stable softmax"""
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

# 3-class example
scores = np.array([[2.0, 1.0, 0.5]])
probs = softmax(scores)
print("Softmax probabilities:", probs)
print("Predicted class:", np.argmax(probs))

**One-vs-Rest (OvR)**: Train K binary classifiers. At inference: pick class with highest score.
**Softmax** gives proper probability distribution.

## Q4. Evaluation Metrics

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc

y_true = np.array([1,0,1,0,0,1])
y_pred = np.array([1,0,0,1,0,1])
y_proba = np.array([0.9, 0.2, 0.4, 0.8, 0.6, 0.1])

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred))

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_true, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

## Q5. Deep Intuition

**Logistic Regression in plain English (5 steps):**
1. Take raw input features.
2. Compute weighted sum (linear layer).
4. Pass through sigmoid → probability between 0 and 1.
5. Threshold at 0.5 (or tune) to get class prediction.
5. Train by minimizing cross-entropy loss (with optional regularization).

**Overfitting diagnosis & fixes** (99% train / 51% test):
- Stronger regularization (`λ` or lower `C`)
- More training data / augmentation
- Feature selection / dimensionality reduction

**Logistic Regression vs Linear SVM**
- Both give linear decision boundaries.
- LR optimizes probabilistic log-loss + regularization.
- SVM optimizes hinge loss + margin maximization.
- Prefer LR when you want calibrated probabilities.
- Prefer SVM when you care more about margin / robustness to outliers.